# 📝 텍스트 전처리·분석 과제 LV2 정답 — TF-IDF·n-gram·동시 출현 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day11_텍스트전처리_분석/data/`·`../../day11_텍스트전처리_분석/images/` 입니다.
- 그래프 문제(8)와 서술형(1)은 자가채점이 없습니다(문제 7 은 히트맵이지만 **단어쌍 값은 채점**합니다).
- TF-IDF **점수 자체는 채점하지 않습니다** — 라이브러리 버전에 따라 소수점이 흔들릴 수 있어 **순위·포함 여부·개수·모양(shape)** 으로만 채점합니다. 빈도·동시 출현은 정수라 정확히 채점합니다.

아래 셀들을 먼저 실행해 라이브러리와 **표준 전처리 파이프라인**을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

정제(`clean_text`)와 토큰화(`tokenize`)는 앞에서 직접 만들어 봤으니, 여기서는 **완성본을 그대로 제공**합니다. 이 과제의 초점은 그 위에서 하는 **분석**입니다.

In [ ]:
# [제공 코드] 표준 전처리 파이프라인 — 정제 함수와 토큰화 함수, 일반 불용어입니다.
def clean_text(text):
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', str(text))   # 특수문자·이모티콘 제거
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)              # 반복 문자 축약
    text = re.sub(r'\s+', ' ', text)                        # 공백 정규화
    return text.strip()

def tokenize(text, stopwords):
    """정제 후 명사·형용사·동사(2글자 이상) 중 불용어가 아닌 것만 돌려줍니다."""
    return [token.form for token in kiwi.tokenize(clean_text(text))
            if token.tag.startswith(('NN', 'VA', 'VV'))
            and len(token.form) > 1 and token.form not in stopwords]

with open('../../day11_텍스트전처리_분석/data/stopwords_ko.json', encoding='utf-8') as f:
    general_stopwords = set(json.load(f))
print("일반 불용어 개수:", len(general_stopwords))

문제 7 에서 쓸 **동시 출현 네트워크 그리기 함수**도 미리 제공합니다. 네트워크 그리기 자체는 이 단원의 학습 목표가 아니니 **호출만** 하면 됩니다(어떤 단어쌍을 넘길지 고르는 것이 여러분의 몫입니다).

In [ ]:
# [제공 코드] 동시 출현 네트워크를 그려 주는 함수입니다 — 내용은 이해하지 않아도 됩니다. 호출만 하세요.
import networkx as nx

def draw_cooccurrence_network(pairs, top_n=25, title='동시 출현 네트워크',
                              node_color='#6aa9e9', save_path=None):
    """pairs = [(단어1, 단어2, 동시출현횟수), ...] 를 네트워크로 그립니다."""
    G = nx.Graph()
    for w1, w2, cnt in pairs[:top_n]:
        G.add_edge(w1, w2, weight=int(cnt))
    strength = {n: sum(d['weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes()}
    lo, hi = min(strength.values()), max(strength.values())
    rng = (hi - lo) or 1
    sizes = [300 + (strength[n] - lo) / rng * 2200 for n in G.nodes()]
    mx = max(d['weight'] for _, _, d in G.edges(data=True))
    widths = [0.5 + G[u][v]['weight'] / mx * 4 for u, v in G.edges()]
    pos = nx.spring_layout(G, k=0.7, seed=42)
    fig, ax = plt.subplots(figsize=(11, 8))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.35, edge_color='#999999', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_color, alpha=0.9,
                           edgecolors='white', linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_family=KOREAN_FONT, font_size=11, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=110, bbox_inches='tight')
    plt.show()
    return G

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe(exclude='number')` 로 범주형 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약·카테고리 분포
preview_news = pd.read_csv('../../day11_텍스트전처리_분석/data/news_headlines.csv')
print("행·열 크기:", preview_news.shape)
print("\n[앞 5행] head()"); display(preview_news.head())
print("\n[열·자료형·결측] info()"); preview_news.info()
print("\n[범주형 요약] describe(exclude='number')"); display(preview_news.describe(exclude='number'))
print("\n[카테고리 분포]"); display(preview_news["category"].value_counts())

## 1. 데이터 살펴보기 (서술형)
**배경**: 리뷰와 달리 뉴스 헤드라인은 **짧고 격식체**입니다. 데이터의 성격이 다르면 전처리·분석에서 주의할 점도 달라집니다. 위 `데이터 살펴보기` 출력을 보고 알게 된 사실을 정리해 보세요.

**요구사항**:
- 아래 서술 셀에 이 데이터에 대한 **관찰 2~3가지**를 문장으로 적으세요.
- 예를 들어: 헤드라인이 몇 개인지, 카테고리(`category`)가 몇 종류이고 균형이 맞는지, 제목(`title`)의 길이·문체가 리뷰와 어떻게 다른지, 눈에 띄는 표기(예: 제목 끝에 붙은 태그 같은 말)가 있는지 등.
- 정답은 하나가 아닙니다. 출력에서 실제로 확인되는 사실이면 됩니다.

> 이 문제는 자가채점(assert)이 없습니다. 아래 서술 셀에 직접 문장을 적고, 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

- 헤드라인은 1200개, 열은 `category`(분야)와 `title`(제목) 두 개다. 결측치는 없다.
- 카테고리는 **스포츠·경제·IT과학·세계 네 종류이고 각각 300개씩 완전 균형**이다. 그래서 카테고리별 비교가 공정하다.
- 제목은 한 줄짜리 짧은 문장이라 리뷰보다 토큰이 훨씬 적다. 대신 `KT`·`SKT` 같은 영문 약어, `…종합` 처럼 **제목 끝에 붙은 편집 태그**가 보인다. 이런 태그는 내용과 무관하니 걸러야 한다.

## 2. 뉴스 도메인 불용어 만들기와 빈도 top10
**배경**: 제공된 `tokenize` 는 **일반 불용어**만 걸러 냅니다. 뉴스 헤드라인에는 `종합`·`게시판` 처럼 **편집 과정에서 붙는 태그**나 `기자`·`속보`·`뉴스` 처럼 **모든 기사에 나오는 말**이 섞여 있습니다. 이런 말은 '이 기사가 무슨 내용인지'를 알려 주지 않으므로, 이 도메인 전용 불용어로 걸러야 합니다.

**요구사항**:
- `data/news_headlines.csv` 를 불러와 `news` 변수에 담으세요.
- 다음 5개 단어를 원소로 갖는 **집합** `domain_stopwords` 를 만드세요 — `종합`, `게시판`, `기자`, `속보`, `뉴스`
- 일반 불용어와 도메인 불용어를 **합친 집합**을 `stopwords` 변수에 담으세요(집합끼리는 `|` 로 합칠 수 있습니다).
- 헤드라인 1200개를 각각 `tokenize(제목, stopwords)` 로 토큰화해, **토큰 리스트들의 리스트**를 변수 `news_tokens`에 담으세요(길이 1200 — 헤드라인 하나당 토큰 리스트 하나).
- 모든 토큰의 빈도를 `Counter` 로 세어 `counter_news` 변수에 담고, 상위 10개 단어(횟수 제외)를 리스트 `top10` 변수에 담아 출력하세요.

**예시**
```
len(news_tokens)          →  1200
counter_news['개발']       →  41     (가장 많이 나온 단어)
counter_news['감독']       →  36
counter_news['종합']       →  0      (도메인 불용어라 제거됨)
top10                     →  개발, 감독, 출시, 농구, 대표, 금융, 프로, 기술, 작년, 축구
```
<details><summary>힌트</summary>

```text
접근방법:
- 일반 불용어 집합에 뉴스 전용 불용어 집합을 합쳐 하나의 불용어 집합을 만든다.
- 헤드라인마다 제공된 토큰화 함수를 부르고, 그 결과를 리스트로 모은다.
- 모든 토큰을 평평하게 펼쳐 Counter 로 센다.

세부구현:
1. csv 를 읽어 `news` 변수에 담는다
2. 중괄호로 도메인 불용어 집합을 만들고, 일반 불용어와 합집합을 취해 `stopwords` 변수에 담는다
3. 제목 열을 순회하며 tokenize 를 호출한 결과를 news_tokens 에 모은다
4. 이중 반복(또는 체이닝)으로 모든 토큰을 펼쳐 Counter 에 넣는다
5. 상위 10개에서 단어만 뽑아 `top10` 변수에 담는다
```

</details>

In [ ]:
news = pd.read_csv('../../day11_텍스트전처리_분석/data/news_headlines.csv')

domain_stopwords = {'종합', '게시판', '기자', '속보', '뉴스'}
stopwords = general_stopwords | domain_stopwords

news_tokens = [tokenize(title, stopwords) for title in news['title']]
counter_news = Counter(word for tokens in news_tokens for word in tokens)
top10 = [word for word, _ in counter_news.most_common(10)]

print("헤드라인 수:", len(news_tokens))
print("빈도 top10:")
for word, count in counter_news.most_common(10):
    print(f"  {word}: {count}")

In [ ]:
# [자가채점]
assert len(news_tokens) == 1200
assert domain_stopwords == {'게시판', '기자', '뉴스', '속보', '종합'}
assert counter_news['개발'] == 41
assert counter_news['감독'] == 36
assert counter_news['출시'] == 33
for word in ['종합', '게시판', '기자']:
    assert counter_news[word] == 0
assert len(top10) == 10
for word in ['개발', '감독', '출시', '농구', '대표']:
    assert word in top10
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: 도메인 불용어는 **데이터를 보고 사람이 정합니다**. 도메인 불용어를 넣기 전 빈도 1위는 `종합`(51회)이었는데, 이건 '종합 기사'라는 편집 태그일 뿐 내용이 아닙니다. 걸러 내자 `개발`·`감독`·`출시` 처럼 **실제 기사 내용**을 말해 주는 단어가 상위로 올라옵니다.
- **흔한 실수**: `tokenize(title)` 처럼 불용어 인자를 빼면 에러가 납니다. 이 함수는 불용어 집합을 두 번째 인자로 받습니다. 또 `news_tokens` 를 평평한 리스트로 만들면 문제 3 에서 문서별로 다시 묶어야 합니다 — **헤드라인당 하나의 리스트**를 유지하세요.
- **대안**: 집합 합치기는 `general_stopwords | domain_stopwords` 또는 `general_stopwords.union(domain_stopwords)` 둘 다 됩니다.

## 3. TF-IDF 벡터화와 한 헤드라인의 핵심어
**배경**: 빈도(Counter)는 '많이 나온 말'만 알려 줍니다. **TF-IDF** 는 여기에 '**이 문서에만** 나오는 말인가'를 곱해, 그 문서를 대표하는 **핵심어**를 골라 줍니다. 모든 헤드라인을 TF-IDF 벡터로 바꾸고, 한 헤드라인의 핵심어를 읽어 봅시다.

**요구사항**:
- 문제 2 의 `news_tokens`(토큰 리스트들의 리스트)를 **공백으로 이어 붙인 문자열 리스트** `docs` 로 만드세요(길이 1200). `TfidfVectorizer` 는 이미 토큰화된 리스트가 아니라 **문자열**을 받기 때문입니다.
- `TfidfVectorizer` 를 `min_df=5`, `max_df=0.85`, `token_pattern=r'\S+'` 로 만들어 `tfidf` 변수에 담고, `docs` 로 학습·변환한 결과를 `X` 변수에 담으세요. 두 값은 **어휘를 양쪽에서 잘라 내는 안전장치**입니다.
  - `min_df=5` — **5개 미만의 문서에만** 나오는 희귀 단어는 버립니다(오타·1회성 고유명사).
  - `max_df=0.85` — **전체 문서의 85%를 넘게** 나오는 단어를 문서 빈도 기준으로 버립니다. 고빈도 불용어 **후보를 거르는 보조 필터**이며 의미는 판단하지 못하므로 제거된 어휘를 사람이 확인해야 합니다 (비율 대신 정수를 주면 문서 개수로 해석됩니다).
- 어휘 목록을 `tfidf.get_feature_names_out()` 으로 얻어 `terms` 변수에 담으세요.
- `X.shape` 와 어휘 수를 출력하세요 — 헤드라인 1200개 × 어휘 268개입니다.
- **6번 헤드라인**에서 TF-IDF 점수가 0보다 큰 단어만 골라, 점수가 **높은 순서로 상위 3개**를 리스트 `doc6_top`에 담아 출력하세요. 문서에 없는 0점 어휘를 핵심어로 뽑으면 안 됩니다.
  - 희소 행렬의 한 행은 `X[6].toarray()[0]` 으로 1차원 배열로 꺼낼 수 있습니다.
  - 큰 값부터의 순서는 `numpy` 의 정렬 인덱스 함수(`argsort`)로 얻습니다(오름차순이니 뒤집어야 합니다).

**예시**
```
X.shape        →  (1200, 268)
len(terms)     →  268
6번 헤드라인    →  'KB손보·서울대금융연 건강·보험·금융 공동연구'
doc6_top[0]    →  '금융'   (이 헤드라인의 1순위 핵심어)
모든 핵심어 점수 →  0보다 큼
```
<details><summary>힌트</summary>

```text
접근방법:
- 토큰 리스트를 공백으로 이어 문자열 문서로 만든 뒤 TF-IDF 벡터라이저에 학습시킨다.
- 한 문서의 행을 배열로 꺼내 점수가 큰 순서의 인덱스를 구하고, 그 인덱스로 어휘 이름을 찾는다.

세부구현:
1. 각 토큰 리스트를 공백으로 join 해 docs 를 만든다
2. TfidfVectorizer 를 min_df=5·max_df=0.85·공백 기준 토큰 패턴으로 만들고 docs 에 fit_transform 해 X 를 얻는다
3. get_feature_names_out 으로 어휘 배열 terms 를 얻고 X 의 shape 를 출력한다
4. X 의 6번 행을 1차원 배열로 바꾸고 0보다 큰 위치만 고른다
5. 양수 위치의 점수를 큰 순서로 정렬해 앞 3개 어휘를 `doc6_top` 변수에 담는다
```

</details>

In [ ]:
docs = [' '.join(tokens) for tokens in news_tokens]

tfidf = TfidfVectorizer(min_df=5, max_df=0.85, token_pattern=r'\S+')
X = tfidf.fit_transform(docs)
terms = tfidf.get_feature_names_out()
print("TF-IDF 행렬 모양:", X.shape)
print("어휘 수:", len(terms))

row = X[6].toarray()[0]
nonzero_idx = np.flatnonzero(row > 0)
order = nonzero_idx[np.argsort(-row[nonzero_idx])]
doc6_top = [terms[i] for i in order[:3]]
print("\n6번 헤드라인:", news.loc[6, "title"])
print("핵심어 top3:", doc6_top)

In [ ]:
# [자가채점]
assert X.shape == (1200, 268)
assert len(terms) == 268
assert len(docs) == 1200
assert len(doc6_top) == 3
assert len(set(doc6_top)) == 3
assert doc6_top[0] == '금융'
assert set(doc6_top[1:]).issubset({'공동', '보험', '서울대'})
term_index = {word: i for i, word in enumerate(terms)}
check_row = X[6].toarray()[0]
assert all(check_row[term_index[word]] > 0 for word in doc6_top)
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `TfidfVectorizer` 는 문자열을 받아 스스로 단어를 자릅니다. 한국어는 공백으로 자르면 조사가 붙어 버리므로, **우리가 형태소로 토큰화한 결과를 공백으로 다시 이어 붙여** 넘깁니다 — 이러면 벡터라이저가 우리가 정한 단어 그대로 씁니다.
- **TF-IDF 읽는 법**: 6번 헤드라인에서 `금융`이 가장 높은 핵심어입니다. 문서에 실제로 등장해 점수가 0보다 큰 단어만 후보로 삼았으므로, 다른 헤드라인의 단어가 0점으로 끼어들지 않습니다. **여러 문서에 흔히 나오는 단어는 점수가 깎이기** 때문입니다(그게 IDF 의 역할). 그래서 TF-IDF 는 '이 문서다운' 단어를 골라 줍니다.
- **흔한 실수**: `X` 는 **희소 행렬**이라 `X[0]` 을 바로 정렬할 수 없습니다. `.toarray()[0]` 으로 1차원 배열로 바꾼 뒤 다루세요. 또 `min_df=5` 를 빼면 어휘가 수천 개로 늘어나 shape 가 달라집니다.
- **`max_df` 이야기**: 이 데이터에서는 `max_df=0.85` 를 넣어도 **어휘 수가 268개로 그대로**입니다 — 헤드라인이 짧아 어떤 단어도 전체의 85%에 나올 만큼 흔하지 않기 때문이죠(가장 흔한 `개발` 도 41번, 약 3%). 그래도 습관처럼 넣어 둡니다. 긴 문서나 정형 문서(계약서·공고문처럼 상투어가 반복되는 글)에서는 `max_df` 가 실제로 여러 단어를 잘라 냅니다. **비율이 아니라 정수**(예: `max_df=1000`)를 주면 '문서 1000개를 넘게 나오면 버림'으로 해석됩니다.

## 4. 카테고리별 평균 TF-IDF — 스포츠 기사는 무슨 말을 쓰나
**배경**: 문서 하나가 아니라 **카테고리 전체**의 핵심어가 궁금할 때가 있습니다. 그 카테고리에 속한 문서들의 TF-IDF 벡터를 **평균**내면, 그 분야에서 점수가 높은 단어를 뽑을 수 있습니다.

**요구사항**:
- `X` 를 밀집 배열로 바꿔 `X_dense` 변수에 담으세요(`.toarray()`). 모양은 문제 3 과 같습니다.
- `news['category']` 가 `'스포츠'` 인 행만 고르는 **불리언 마스크**를 `sports_mask` 변수에 담고(뒤 문제에서도 다시 씁니다), 그 마스크로 고른 스포츠 문서들의 TF-IDF 벡터를 **열 방향으로 평균**(`axis=0`)내어 `sports_mean` 변수에 담으세요(길이는 어휘 수와 같습니다).
- `sports_mean` 이 큰 순서로 **상위 10개 단어**를 `sports_simple` 리스트에 담아 출력하세요.

**예시**
```
sports_mean.shape   →  (268,)      (어휘마다 평균 점수 하나)
sports_simple[0]    →  '감독'      (스포츠 문서 평균 1위)
sports_simple       →  감독, 농구, 프로, 축구, 영입, 선수, 리그, 대표, 코치, 여자
```

> 목록을 눈으로 보세요. `감독`·`농구`·`축구` 는 확실히 스포츠답습니다. 그런데 **`대표`** 는? '국가대표'의 대표일 수도 있지만 '대표 이사'·'각국 대표' 처럼 **다른 분야 기사에도 흔히 나오는 말**입니다. 이 문제를 다음 문제에서 해결합니다.
<details><summary>힌트</summary>

```text
접근방법:
- 카테고리가 스포츠인 행만 골라 TF-IDF 행들을 세로로 평균낸다.
- 평균 점수가 큰 순서의 인덱스를 구해 어휘 이름으로 바꾼다.

세부구현:
1. X 를 밀집 배열로 바꿔 `X_dense` 변수에 담는다
2. category 열이 스포츠와 같은지 비교해 불리언 마스크를 만들어 `sports_mask` 변수에 담는다
3. 마스크로 고른 행들을 axis=0 방향으로 평균내 `sports_mean` 변수에 담는다
4. argsort 를 뒤집어 상위 10개 인덱스를 얻고, terms 로 단어 이름을 찾아 `sports_simple` 변수에 담는다
```

</details>

In [ ]:
X_dense = X.toarray()

sports_mask = (news['category'] == '스포츠').values
sports_mean = X_dense[sports_mask].mean(axis=0)
sports_simple = [terms[i] for i in np.argsort(sports_mean)[::-1][:10]]

print("스포츠 문서 수:", int(sports_mask.sum()))
print("평균 TF-IDF 상위 10개:", sports_simple)

In [ ]:
# [자가채점]
assert X_dense.shape == (1200, 268)
assert int(sports_mask.sum()) == 300
assert sports_mean.shape == (268,)
assert len(sports_simple) == 10
assert sports_simple[0] == '감독'
for word in ['농구', '프로', '축구', '대표']:
    assert word in sports_simple
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 불리언 마스크로 행을 고르고 `axis=0` 평균을 내면 **어휘별 평균 점수 벡터**가 나옵니다(`axis=1` 로 하면 문서별 평균이 나와 전혀 다른 뜻이 됩니다).
- **한계**: 이 목록은 '스포츠에서 점수가 높은 단어'일 뿐, '**스포츠에서만** 점수가 높은 단어'가 아닙니다. `대표`(8위)가 그 증거예요 — 세계·경제 기사에도 자주 나오는 말이 섞여 들어왔습니다. 이런 **공유 노이즈**를 걷어내는 방법이 다음 문제입니다.
- **흔한 실수**: `X` 가 희소 행렬이라 `X[mask]` 가 뜻대로 안 될 수 있습니다. `.toarray()` 로 밀집 배열을 만든 뒤 마스킹하세요(어휘 268개라 메모리 부담이 없습니다).

## 5. 차이 기반 특징어 — 그 분야'만'의 단어 찾기
**배경**: 문제 4 의 단순 평균에는 **모든 분야에 흔한 단어**가 섞입니다. 해결책은 간단합니다 — **그 카테고리의 평균에서 나머지 전체의 평균을 빼는 것**입니다. 모든 분야에 고르게 나오는 단어는 빼기에서 상쇄되어 0 에 가까워지고, **그 분야에서만 두드러진 단어**만 큰 값으로 남습니다.

**요구사항**:
- 문제 4 의 `X_dense`·`terms` 를 이어 씁니다.
- **스포츠**: 스포츠 문서 평균(`sports_mean`)에서 **스포츠가 아닌 문서 전체의 평균**을 뺀 배열을 변수 `sports_diff`에 담으세요(마스크의 반대는 `~mask` 로 얻습니다).
- `sports_diff` 가 큰 순서로 상위 10개 단어를 `sports_feature` 리스트에 담으세요.
- 같은 방식으로 **IT과학** 에 대해서도 상위 10개 단어를 `it_feature` 리스트에, 단순 평균 상위 10개를 `it_simple` 리스트에 담으세요.
- 문제 8 의 워드클라우드에 쓸 수 있도록, 스포츠의 `(단어, 차이점수)` 쌍을 **점수 내림차순으로** 정렬한 리스트를 `sports_diff_scores` 변수에 담으세요(상위 30개 이상 들어 있으면 됩니다).
- 단순 평균(문제 4)과 차이 기반 목록을 **나란히 출력해 무엇이 달라졌는지** 확인하세요.

**예시**
```
sports_simple  (단순)  →  감독, 농구, 프로, 축구, 영입, 선수, 리그, 대표, 코치, 여자
sports_feature (차이)  →  감독, 농구, 프로, 축구, 영입, 선수, 리그, 코치, 여자, 월드컵
   → 다른 분야에도 흔한 '대표' 가 빠지고, 스포츠 고유어 '월드컵' 이 올라온다

it_simple      (단순)  →  개발, 출시, 기술, 과학, 삼성, 발견, 우주, 교수, 효과, 로봇
it_feature     (차이)  →  개발, 출시, 기술, 과학, 발견, 교수, 로봇, 세포, 삼성, 우주
   → 다른 분야에도 흔한 '효과' 가 빠지고, IT과학 고유어 '세포' 가 올라온다
```
<details><summary>힌트</summary>

```text
접근방법:
- 그 카테고리의 평균 벡터와, 그 카테고리가 아닌 문서들의 평균 벡터를 각각 구한다.
- 두 벡터를 빼면 '그 분야에서만 두드러진 정도' 가 된다. 큰 순서로 정렬해 단어를 읽는다.

세부구현:
1. 카테고리 마스크를 만들고, 물결표로 반대 마스크를 만든다
2. 각각 axis=0 평균을 내어 안쪽 평균·바깥쪽 평균을 얻는다
3. 안쪽 평균에서 바깥쪽 평균을 빼 차이 배열을 만든다
4. argsort 를 뒤집어 상위 10개 인덱스를 얻고 terms 로 단어를 찾는다
5. IT과학도 같은 절차를 반복한다(단순 평균 목록도 함께 만든다)
6. 스포츠는 (단어, 차이점수) 쌍을 점수 내림차순으로 정렬해 따로 보관한다
```

</details>

In [ ]:
sports_other_mean = X_dense[~sports_mask].mean(axis=0)
sports_diff = sports_mean - sports_other_mean
sports_order = np.argsort(sports_diff)[::-1]
sports_feature = [terms[i] for i in sports_order[:10]]
sports_diff_scores = [(terms[i], float(sports_diff[i])) for i in sports_order]

it_mask = (news['category'] == 'IT과학').values
it_mean = X_dense[it_mask].mean(axis=0)
it_other_mean = X_dense[~it_mask].mean(axis=0)
it_diff = it_mean - it_other_mean
it_simple = [terms[i] for i in np.argsort(it_mean)[::-1][:10]]
it_feature = [terms[i] for i in np.argsort(it_diff)[::-1][:10]]

print("[스포츠] 단순 평균:", sports_simple)
print("[스포츠] 차이 기반:", sports_feature)
print("[IT과학] 단순 평균:", it_simple)
print("[IT과학] 차이 기반:", it_feature)

In [ ]:
# [자가채점]
assert sports_diff.shape == (268,)
assert len(sports_feature) == 10 and len(it_feature) == 10 and len(it_simple) == 10
assert sports_feature[0] == '감독'
assert it_feature[0] == '개발'
# 차이 기반은 단순 평균과 달라야 한다 — 공유 노이즈가 걷힌 결과
assert set(sports_feature) != set(sports_simple)
assert set(it_feature) != set(it_simple)
assert '대표' in sports_simple and '대표' not in sports_feature
assert '월드컵' in sports_feature and '월드컵' not in sports_simple
assert '효과' in it_simple and '효과' not in it_feature
assert '세포' in it_feature and '세포' not in it_simple
assert len(sports_diff_scores) >= 30
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: `그 분야 평균 − 나머지 평균` 은 **대비(contrast)** 를 재는 가장 단순하고 강력한 방법입니다. `대표` 처럼 여러 분야에 고르게 나오는 단어는 두 평균이 비슷해 차이가 작아지고, `월드컵`·`세포` 처럼 한 분야에만 몰린 단어는 나머지 평균이 거의 0 이라 차이가 그대로 남습니다.
- **결과 읽기**: 뉴스 카테고리는 원래도 잘 구분되는 데이터라 두 목록의 **상위권은 거의 같습니다**(`감독`·`농구`·`프로`는 애초에 스포츠 고유어니까요). 차이는 **경계(8~10위)에서 드러납니다** — 거기서 공유 노이즈(`대표`·`효과`)가 밀려나고 고유어(`월드컵`·`세포`)가 올라옵니다. 분야 구분이 희미한 데이터일수록 이 효과는 훨씬 커집니다.
- **흔한 실수**: 나머지 평균을 '전체 평균'으로 계산하면(그 분야를 포함한 채) 대비가 약해집니다. 반드시 **그 분야를 뺀 나머지**(`~mask`)로 평균을 내세요.
- **대안**: 빼기 대신 나누기(비율)를 쓸 수도 있지만, 분모가 0 에 가까운 단어에서 값이 폭발합니다. 빼기가 안전합니다.

## 6. n-gram — 붙어 다니는 두 단어 찾기
**배경**: 단어를 하나씩 세면 `프로`와 `농구`가 따로 놉니다. 하지만 이 둘은 거의 항상 **붙어서** `프로 농구`로 쓰이죠. 연속한 두 단어를 하나로 세는 것이 **bigram**(2-gram)입니다.

**요구사항**:
- 문제 3 의 `docs` 를 이어 씁니다.
- `CountVectorizer` 를 `ngram_range=(2, 2)`, `min_df=3`, `token_pattern=r'\S+'` 로 만들어 `bigram_vec` 변수에 담고, `docs` 로 학습·변환한 결과를 `X_bigram` 변수에 담으세요. (`ngram_range=(2, 2)` = 두 단어 묶음만, `min_df=3` = 3개 문서 미만에 나오는 묶음은 버림.)
- bigram 이름을 `bigram_vec.get_feature_names_out()` 으로 얻어 `bigram_terms` 변수에 담으세요(38개가 나옵니다).
- 각 bigram 의 **전체 등장 횟수**를 구하세요 — `X_bigram` 을 **열 방향으로 합**(`axis=0`)하면 됩니다. 희소 행렬의 합은 `np.asarray(...).ravel()` 로 1차원 배열로 펴 주세요.
- 횟수가 큰 순서로 **상위 10개**를 `(bigram, 횟수)` 튜플 리스트로 만들어 `bigram_top10` 변수에 담아 출력하세요. 횟수는 **정수**여야 합니다.

**예시**
```
len(bigram_terms)   →  38
bigram_top10[0]     →  ('프로 농구', 15)
bigram_top10 안에    →  ('영업 이익', 9) · ('아시안 게임', 9) · ('달러 환율', 6) 등이 들어 있다
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 단어 묶음만 세는 벡터라이저를 만들어 문서에 학습시킨다.
- 문서-묶음 행렬을 세로로 합치면 묶음별 총 등장 횟수가 된다. 큰 순서로 10개를 고른다.

세부구현:
1. CountVectorizer 를 ngram_range 와 min_df 를 지정해 만든다
2. docs 에 fit_transform 해 X_bigram 을 얻고, get_feature_names_out 으로 bigram_terms 를 얻는다
3. X_bigram 을 axis=0 으로 합해 1차원 정수 배열로 편다
4. argsort 를 뒤집어 상위 10개 인덱스를 얻는다
5. 각 인덱스의 (bigram 이름, 정수 횟수) 튜플을 `bigram_top10` 변수에 담는다
```

</details>

In [ ]:
bigram_vec = CountVectorizer(ngram_range=(2, 2), min_df=3, token_pattern=r'\S+')
X_bigram = bigram_vec.fit_transform(docs)
bigram_terms = bigram_vec.get_feature_names_out()

bigram_counts = np.asarray(X_bigram.sum(axis=0)).ravel()
bigram_order = np.argsort(bigram_counts)[::-1][:10]
bigram_top10 = [(bigram_terms[i], int(bigram_counts[i])) for i in bigram_order]

print("bigram 개수:", len(bigram_terms))
for phrase, count in bigram_top10:
    print(f"  {phrase}: {count}")

In [ ]:
# [자가채점]
assert len(bigram_terms) == 38
assert bigram_top10[0] == ('프로 농구', 15)
assert len(bigram_top10) == 10
bigram_dict = dict(bigram_top10)
assert bigram_dict['영업 이익'] == 9
assert bigram_dict['아시안 게임'] == 9
assert bigram_dict['달러 환율'] == 6
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `ngram_range=(2, 2)` 는 **연속한 두 단어**만 셉니다. 우리가 이미 형태소로 토큰화해 공백으로 이어 붙였으니, 여기서 말하는 '두 단어'는 **두 형태소**입니다(그래서 `프로 농구`처럼 의미 있는 묶음이 나옵니다).
- **흔한 실수**: `X_bigram.sum(axis=0)` 은 희소 행렬 전용 자료형(matrix)을 돌려줘 그대로 정렬하면 이상하게 동작합니다. `np.asarray(...).ravel()` 로 평범한 1차원 배열로 바꾸세요. 또 횟수를 `int(...)` 로 감싸지 않으면 numpy 정수가 되어 튜플 비교가 예상과 달라질 수 있습니다.
- **읽는 법**: `프로 농구`(15) 처럼 항상 붙어 다니는 묶음은 사실상 **하나의 단어**입니다. 이런 묶음을 발견하면 토큰화 단계에서 한 단어로 합치는 것도 좋은 전처리입니다.
- **대안**: `min_df=3` 을 낮추면 bigram 이 폭증하고(한 번만 나온 우연한 조합까지), 높이면 몇 개 안 남습니다. 헤드라인이 짧으니 3 정도가 적당합니다.

## 7. 동시 출현 — 한 기사에 함께 나오는 단어쌍
**배경**: bigram 은 **바로 붙어 있는** 두 단어만 잡습니다. 하지만 `영업`과 `작년` 처럼 **한 기사 안에 떨어져 있어도 함께 등장하는** 단어쌍도 중요합니다. 이것이 **동시 출현**(co-occurrence)입니다.

**요구사항**:
- 문제 3 의 `docs` 를 이어 씁니다.
- `CountVectorizer` 를 `binary=True`, `min_df=10`, `max_df=0.85`, `token_pattern=r'\S+'` 로 만들어 `co_vec` 변수에 담고, `docs` 로 학습·변환한 결과를 `X_co` 변수에 담으세요.
  - `binary=True` — '몇 번 나왔나'가 아니라 **'나왔나(1) 안 나왔나(0)'** 만 기록합니다. 동시 출현은 횟수가 아니라 **함께 등장한 문서 수**를 세는 것이니까요.
  - `min_df=10` — 10개 미만의 문서에만 나오는 드문 단어는 버립니다.
  - `max_df=0.85` — **전체 문서의 85%를 초과해** 등장하는 단어는 버립니다. 정확히 85%면 유지됩니다. 어디에나 나오는 단어는 아무하고나 함께 등장해 연관 분석을 흐리므로, **문서빈도를 기준으로 고빈도 불용어 후보를 제외하는** 안전장치입니다. 기준은 출현 **'횟수'가 아니라 그 단어가 등장한 '문서 수'** 이며, 어떤 단어를 의미상 불용어로 볼지는 결과를 확인해 사람이 판단해야 합니다.
- 어휘를 `co_vec.get_feature_names_out()` 으로 얻어 `terms_co` 변수에 담으세요(70개).
- 문서-단어 행렬의 **전치 행렬과 원 행렬을 곱한 뒤 밀집 배열로 변환**해 `matrix` 변수에 담고, NumPy의 대각선 채우기 함수로 **대각선(자기 자신과의 동시 출현)을 0**으로 만드세요.
  - 결과 행렬의 `(i, j)` 칸은 '단어 i 와 단어 j 가 **함께 등장한 문서 수**' 가 됩니다.
- NumPy의 상삼각 인덱스 함수에 `k=1`을 주어 대각선을 제외한 **위쪽 삼각형**만 고른 뒤, 값이 큰 순서로 **상위 25개 단어쌍**을 `(단어1, 단어2, 문서 수)` 튜플 리스트로 만들어 `top_pairs` 변수에 담으세요. 문서 수는 **정수**입니다. (같은 쌍이 두 번 세어지는 것을 막기 위해 위쪽 삼각형만 봅니다.) 상위 5쌍을 출력해 확인하세요.
- **히트맵**: 동시 출현이 많은 **상위 15개 단어**에 대한 **seaborn 히트맵**을 그리세요(각 단어의 동시 출현 총합 `matrix.sum(axis=1)` 이 큰 순서로 15개, `annot=True`, `fmt='d'`, `cmap='YlGnBu'`, `fig, ax = plt.subplots(figsize=(9, 7))`, `sns.heatmap(..., ax=ax)`).
- **네트워크**: 위에서 만든 `top_pairs` 를 **제공 함수** `draw_cooccurrence_network` 에 넘겨 네트워크를 그리세요(`top_n=25`, `title` 은 자유롭게). 함수 내부는 이해하지 않아도 됩니다 — **어떤 단어쌍을 넘길지 고르는 것**이 여러분의 몫입니다. 단어는 **점(노드)**, 함께 나온 관계는 **선(엣지)** 이 되고, 많이 엮인 단어일수록 점이 커집니다.

**예시**
```
len(terms_co)   →  70
matrix.shape    →  (70, 70)
len(top_pairs)  →  25
top_pairs[0]    →  ('농구', '프로', 15)      (같은 기사에 함께 나온 문서 수)
top_pairs 안에   →  ('영업', '이익', 9) · ('이익', '작년', 9) 같은 경제 기사 짝이 보인다
```

> 히트맵·네트워크는 아래 완성 그래프와 같은 모양이면 됩니다. **단어쌍(`top_pairs`)은 자가채점**합니다.
<details><summary>힌트</summary>

```text
접근방법:
- 등장 여부만 0/1 로 기록하는 벡터라이저로 문서-단어 행렬을 만든다.
- 그 행렬의 전치와 자기 자신을 곱하면 단어×단어 동시 출현 행렬이 된다. 대각선은 자기 자신이니 0 으로 지운다.
- 위쪽 삼각형의 값만 모아 큰 순서로 정렬하고, 그 위치의 두 단어 이름을 찾는다.
- 동시 출현 총합이 큰 15개 단어만 잘라 부분 행렬을 만들고 히트맵으로 그린다.
- 네트워크는 제공 함수에 단어쌍 리스트를 넘기기만 하면 된다.

세부구현:
1. CountVectorizer 를 binary·min_df·max_df 를 지정해 만들고 docs 에 학습·변환한다
2. 전치 행렬과 원 행렬의 행렬곱을 배열로 바꿔 `matrix` 변수에 담고 대각선을 0 으로 채운다
3. 위쪽 삼각형의 행·열 인덱스를 얻고 그 위치의 값들을 모은다
4. 값이 큰 순서로 25개를 골라 (단어1, 단어2, 정수 값) 튜플 리스트 top_pairs 를 만든다
5. 행별 합이 큰 15개 인덱스를 골라 부분 행렬과 라벨을 만들고, 새 그림에 히트맵을 그린다
6. 제공된 네트워크 함수에 top_pairs 를 넘겨 호출한다
```

</details>

> **완성 그래프(정답)** — 아래 두 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv2_q7_heatmap.png" width="620">

<img src="../../day11_텍스트전처리_분석/images/과제/lv2_q7_network.png" width="700">

In [ ]:
co_vec = CountVectorizer(binary=True, min_df=10, max_df=0.85, token_pattern=r'\S+')
X_co = co_vec.fit_transform(docs)
terms_co = co_vec.get_feature_names_out()

matrix = (X_co.T @ X_co).toarray()   # 단어×단어 동시 출현 행렬
np.fill_diagonal(matrix, 0)          # 자기 자신과의 동시 출현은 제외

rows, cols = np.triu_indices_from(matrix, k=1)   # 위쪽 삼각형만(중복 방지)
values = matrix[rows, cols]
pair_order = np.argsort(values)[::-1][:25]
top_pairs = [(terms_co[rows[i]], terms_co[cols[i]], int(values[i])) for i in pair_order]

print("어휘 수:", len(terms_co), "| 행렬 모양:", matrix.shape)
print("동시 출현 상위 5쌍:")
for word1, word2, count in top_pairs[:5]:
    print(f"  {word1} + {word2}: {count}개 기사")

totals = matrix.sum(axis=1)
top15 = np.argsort(-totals)[:15]
labels = [terms_co[i] for i in top15]
sub = matrix[np.ix_(top15, top15)]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sub, xticklabels=labels, yticklabels=labels, cmap='YlGnBu', annot=True, fmt='d', ax=ax)
ax.set_title('뉴스 헤드라인 단어 동시 출현 (상위 15개 단어)')
plt.show()

draw_cooccurrence_network(top_pairs, top_n=25,
                          title='뉴스 헤드라인 동시 출현 네트워크 (상위 25쌍)')

In [ ]:
# [자가채점]
assert len(terms_co) == 70
assert matrix.shape == (70, 70)
assert matrix.diagonal().sum() == 0
assert len(top_pairs) == 25
assert top_pairs[0] == ('농구', '프로', 15)
pair_counts = {(word1, word2): count for word1, word2, count in top_pairs}
assert pair_counts[('영업', '이익')] == 9
word_index = {word: i for i, word in enumerate(terms_co)}
assert matrix[word_index['축구'], word_index['대표']] == 7
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: `binary=True` 로 만든 문서×단어 행렬 `X_co` 에서, `X_co.T @ X_co` 의 `(i, j)` 칸은 'i 열과 j 열이 **동시에 1인 문서 수**'가 됩니다(0/1 곱셈이 AND 처럼 동작하니까요). 한 줄의 행렬곱으로 모든 단어쌍을 한 번에 세는 것이 핵심입니다 — 이중 반복문으로 짝을 세면 훨씬 느리고 실수하기 쉽습니다.
- **흔한 실수**: 대각선을 0 으로 지우지 않으면 '자기 자신과의 동시 출현'(= 그 단어가 나온 문서 수)이 가장 큰 값이라 상위권을 전부 차지합니다. 또 위쪽 삼각형만 보지 않으면 `(농구, 프로)` 와 `(프로, 농구)` 가 **같은 쌍인데 두 번** 나옵니다.
- **`max_df` 의 역할**: 이 데이터에서는 `max_df=0.85` 를 넣어도 어휘가 **70개 그대로**입니다 — 가장 흔한 `개발` 조차 1200개 중 41개 문서(약 3%)에만 나오기 때문이죠. 그래도 넣는 이유는, 동시 출현에서 **모든 문서에 나오는 단어 하나가 모든 쌍의 상위권을 독식**해 버리기 때문입니다(그 단어는 누구와도 함께 나오니까요). 도메인 불용어를 빠뜨렸을 때 이 옵션이 **자동으로** 막아 줍니다. 기준은 출현 횟수가 아니라 **그 단어가 등장한 문서 수의 비율**입니다.
- **읽는 법**: 상위 쌍은 그 코퍼스의 **주제 덩어리**를 보여 줍니다 — `농구+프로`(스포츠), `영업+이익`·`이익+작년`(경제 실적 기사). 히트맵의 밝은 칸을 따라가면 어떤 단어들이 한 무리로 묶이는지 눈으로 보입니다.
- **히트맵 vs 네트워크**: 히트맵은 **모든 칸의 값을 정확히** 보여 주지만(숫자를 읽어야 함), 네트워크는 **어떤 단어들이 한 덩어리로 뭉치는지**를 한눈에 보여 줍니다 — 스포츠 덩어리(`프로`·`농구`·`축구`·`대표`)와 경제 덩어리(`영업`·`이익`·`작년`·`대비`·`분기`)가 **따로 떨어진 섬**처럼 나뉘어 보이죠. 정확한 값은 히트맵, 구조는 네트워크입니다.
- **네트워크에 넘길 쌍을 고르는 것이 핵심**: 함수는 넘긴 쌍만 그립니다. 쌍을 너무 많이(예: 200쌍) 넘기면 선이 뒤엉켜 아무것도 안 보이고(흔히 '털뭉치'라 부릅니다), 너무 적게 넘기면 구조가 안 드러납니다. 상위 25쌍 정도가 읽기 좋습니다.
- **bigram 과의 차이**: bigram 은 **붙어 있어야** 세지만, 동시 출현은 **같은 문서 안 어디든** 있으면 셉니다. 그래서 `이익 + 작년`(9) 처럼 떨어져 있는 짝도 잡힙니다.

## 8. 카테고리 특징어 워드클라우드
**배경**: 문제 5 에서 구한 **차이 기반 특징어**는 '스포츠다움'의 정도를 점수로 가진 단어 목록입니다. 이 점수를 그대로 워드클라우드의 크기로 쓰면, **스포츠 분야의 얼굴**을 한 장으로 보여 줄 수 있습니다.

**요구사항**:
- 문제 5 의 `sports_diff_scores`(점수 내림차순 `(단어, 차이점수)` 리스트)에서 **양수인 항목만** 고른 뒤 앞 30개를 `positive_scores` 리스트 변수에 담으세요. 이를 `{단어: 점수}` 딕셔너리로 바꿔 `weights` 변수에 담으세요.
- `WordCloud` 를 만들 때 **`font_path=FONT_PATH` 를 반드시 넣으세요** — 빼면 한글이 □□□ 로 깨집니다. 크기는 `width=800, height=500`, 배경은 `background_color='white'`, `max_words=30` 으로 지정한 WordCloud 객체를 `wc` 변수에 담으세요.
- `wc.generate_from_frequencies(weights)` 결과 이미지를 `cloud` 변수에 담으세요 — **빈도 대신 차이 점수**를 크기로 쓰는 것이 이 문제의 핵심입니다.
- `fig, ax = plt.subplots(figsize=(10, 6))`로 Figure와 Axes를 만든 뒤 `ax.imshow(cloud)`로 표시하고, `ax.axis('off')`와 `ax.set_title(...)`을 적용한 뒤 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프와 같은 모양(`감독`·`농구`·`프로`·`축구` 가 가장 크게 보이는 구름)이면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 양수인 차이 점수만 고른 뒤 상위 30개를 단어→점수 딕셔너리로 바꿔 워드클라우드에 넘긴다.
- 한글 폰트 경로를 반드시 지정한다.

세부구현:
1. sports_diff_scores 에서 score > 0 인 항목만 골라 앞 30개를 positive_scores 리스트 변수에 담는다
2. positive_scores 를 weights 딕셔너리 변수로 바꾼다
3. WordCloud 객체를 wc 변수에 담고 generate_from_frequencies 결과를 cloud 변수에 담는다
4. fig, ax = plt.subplots(...)로 만들고 ax.imshow/axis/set_title을 적용한 뒤 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day11_텍스트전처리_분석/images/과제/lv2_q8_wordcloud.png" width="640">

In [ ]:
positive_scores = [(word, score) for word, score in sports_diff_scores if score > 0][:30]
weights = dict(positive_scores)
wc = WordCloud(font_path=FONT_PATH, width=800, height=500,
               background_color='white', max_words=30)
cloud = wc.generate_from_frequencies(weights)
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(cloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('스포츠 카테고리 특징어 워드클라우드 (차이 기반)')
plt.show()

### 해설 — 문제 8
- **접근법**: `generate_from_frequencies` 는 이름만 '빈도'일 뿐, **양수 점수라면 무엇이든** 크기로 씁니다. 그래서 차이 기반 TF-IDF 점수를 그대로 넘길 수 있습니다. 양수 중 상위 30개만 쓰는 이유는, 차이 점수가 0 근처인 단어(= 모든 분야에 흔한 말)를 애초에 배제하기 위해서입니다.
- **흔한 실수**: 차이 점수에는 **음수**(그 분야에서 오히려 덜 쓰이는 단어)가 들어 있습니다. 전체를 넘기면 에러가 나거나 이상하게 그려지니, **양수만 필터링한 뒤 상위 30개**를 넘기세요. 그리고 `font_path` 를 빼면 글자가 전부 네모(□)가 됩니다.
- **비교**: 문제 4 의 단순 평균으로 워드클라우드를 그리면 `대표` 같은 공유어가 큼직하게 들어옵니다. 차이 기반으로 그리면 **그 분야만의 얼굴**이 남습니다 — 카테고리 비교 리포트에는 차이 기반이 훨씬 낫습니다.